# KelpAdam – UAS Pembelajaran Mesin
## Notebook B – Regresi Prediksi IPK dari Survei Preferensi Belajar (VARK)

**Tujuan:** memprediksi IPK (angka kontinu 0–4) dari jawaban 16 skenario VARK, lalu membandingkan **Linear Regression** dan **Random Forest Regressor** secara adil, dengan baseline sebagai patokan.

| Komponen | Isi |
|---|---|
| Sumber data | `dataset/processed_UTS/KelpAdam_survey_prepared.csv` (hasil notebook UTS, 192 responden) |
| Framework | VARK: 64 indikator 0/1 dari 16 skenario |
| Target y | `IPK` (kolom DL pada data dosen), numerik |
| Pembagian data | train 80% : test 20%, `random_state = 42` |
| Model | Baseline rerata, Linear Regression, Random Forest Regressor |
| Metrik | MAE, RMSE, R² pada test set |

**Alur notebook:** B0 Setup → B1 Data & definisi X/y → B2 Preprocessing & cleaning → B3 EDA → B4 Split & pipeline → B5 Training & pemilihan model → B6 Evaluasi test → B7 Simpan model.

**Batas interpretasi:** model ini hanya memeriksa apakah jawaban survei membantu memperkirakan IPK pada data yang tersedia. Hasilnya bukan bukti bahwa preferensi belajar *menyebabkan* IPK. Karena urutan waktu pengisian survei dan IPK tidak diketahui, model juga tidak diklaim sebagai prediksi dini.

---
# B0 – Setup

Semua library, konstanta, dan path didefinisikan di satu sel supaya notebook bisa dijalankan ulang dari awal (*Restart & Run All*).
- `SEED = 42` dan `TEST_SIZE = 0.2` sesuai spesifikasi UAS dan dipakai di seluruh notebook.
- Path dibuat **relatif terhadap folder proyek**, sama seperti notebook UTS kemarin, sehingga notebook berjalan di laptop anggota mana pun tanpa mengubah kode.
- Versi library dicetak sebagai bukti reproduksibilitas. Versi scikit-learn penting karena model `.joblib` hanya aman dimuat dengan versi yang sama.

In [1]:
import json
import platform
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy import stats
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (GridSearchCV, KFold, cross_validate,
                                     permutation_test_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# --- Konstanta eksperimen (sesuai spesifikasi UAS) ---
SEED = 42          # random_state wajib
TEST_SIZE = 0.2    # train : test = 80 : 20

# --- Path relatif: notebook berada di <folder proyek>/notebook/ ---
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_FILE = PROJECT_DIR / "dataset" / "processed_UTS" / "KelpAdam_survey_prepared.csv"
DATA_OUT  = PROJECT_DIR / "dataset" / "processed_UAS"     # Dataset_regresi.csv
MODEL_DIR = PROJECT_DIR / "models"                        # Model_regresi.joblib
OUT_DIR   = PROJECT_DIR / "figures_and_outputs"           # grafik & tabel (prefix B)
for folder in (DATA_OUT, MODEL_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# --- Tampilan tabel & grafik ---
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
BIRU, ORANYE, ABU = "#2b6cb0", "#dd6b20", "#a0aec0"


def simpan_fig(fig, nama):
    """Simpan grafik ke figures_and_outputs/<nama>.png lalu tampilkan."""
    path = OUT_DIR / f"{nama}.png"
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()
    print("Disimpan:", path.relative_to(PROJECT_DIR))


def simpan_tabel(tabel, nama):
    """Simpan tabel ke figures_and_outputs/<nama>.csv agar angka laporan bisa ditelusuri."""
    path = OUT_DIR / f"{nama}.csv"
    tabel.to_csv(path)
    print("Disimpan:", path.relative_to(PROJECT_DIR))


print("Folder proyek :", PROJECT_DIR)
print("File data     :", DATA_FILE.relative_to(PROJECT_DIR), "| ada?", DATA_FILE.exists())
print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scikit-learn {sklearn.__version__} | scipy {scipy.__version__} | "
      f"matplotlib {matplotlib.__version__} | joblib {joblib.__version__}")

Folder proyek : d:\Kuliah\Semester 5\Machine Learning\KelpAdam_UAS_MachineLearning
File data     : dataset\processed_UTS\KelpAdam_survey_prepared.csv | ada? True
Python 3.12.10 | pandas 3.1.1 | numpy 2.5.5 | scikit-learn 1.9.1 | scipy 1.18.1 | matplotlib 3.11.2 | joblib 1.6.0


---
# B1 – Memuat data & mendefinisikan X dan y

## B1.1 Memuat dataset hasil UTS
Panduan UAS menyatakan bahwa *UAS menggunakan dataset yang dihasilkan pada UTS*. Dataset ini sudah melalui cleaning C1–C10 di notebook UTS (194 → 192 baris) dan berisi item mentah, skor, label, serta 64 indikator VARK. File UTS hanya **dibaca**, tidak diubah.

In [ ]:
df = pd.read_csv(DATA_FILE)
print("Ukuran dataset:", df.shape)

# Memastikan ukuran harus sama dengan yang diharapkan
assert df.shape == (192, 216), "Ukuran berbeda dari output UTS; periksa file di dataset/processed_UTS/"
assert df["ID"].is_unique, "ID tidak unik"

# Menampilkan contoh 5 baris pertama dataset
display(df[["ID", "IPK", "VARK_Q01", "VARK_Q01_V", "VARK_Q01_A", "VARK_Q01_R", "VARK_Q01_K"]].head())

Ukuran dataset: (192, 216)


,ID,IPK,VARK_Q01,VARK_Q01_V,VARK_Q01_A,VARK_Q01_R,VARK_Q01_K
0,1,3.78,"melihat diagram, bagan, peta konsep, atau ilus...",1,1,0,0
1,2,3.83,"melihat diagram, bagan, peta konsep, atau ilus...",1,0,0,1
2,3,3.77,"melihat diagram, bagan, peta konsep, atau ilus...",1,0,0,1
3,4,1.97,"mencoba contoh, simulasi, praktik, atau meliha...",0,0,1,1
4,5,3.54,mendengar penjelasan dan dapat berdiskusi atau...,0,1,0,0
